In [2]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import JSONLoader
from langchain_community.vectorstores import Chroma

from ibm_watson_machine_learning.metanames import GenTextParamsMetaNames as GenParams

import json
import pdfplumber
import os

In [39]:
#  function to extract data from the pdf files
def extract_text_from_multiple_pfds(pdf_paths):
    data=[]
    for path in pdf_paths:
        text=""
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                text+= page.extract_text()+ "\n\n"
        data.append({"url":path, "text":text})
    return data 

In [40]:
pdf_path = ["Data/Engineering_Physics.pdf", 
            "Data/Modern Physics for Scientists and Engineers.pdf", 
            "Data/Tipler_Llewellyn_Physics.pdf"]

In [41]:
extracted_data = extract_text_from_multiple_pfds(pdf_path)

In [42]:
json_path = 'extracted_data_new.json'

with open(json_path, 'w') as file:
    json.dump(extracted_data, file, indent=4)

In [43]:
loader = JSONLoader(file_path='extracted_data_new.json', 
                    jq_schema='.[]', text_content=False)

data = loader.load()

In [44]:
content_list = [json.loads(doc.page_content)['text'] for doc in data]

In [45]:
metadata_list = [json.loads(doc.page_content)['url'] for doc in data]

In [46]:
docs = [{
    "page_content":content, "metadata":metadata} 
    for content, metadata in zip(content_list,metadata_list)
]

In [47]:
docs[2]['metadata']

'Data/Tipler_Llewellyn_Physics.pdf'

In [48]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 20

text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE
                                               , chunk_overlap=CHUNK_OVERLAP)

In [49]:
chunked_docs=[]

for doc in docs:
    chunks = text_splitter.create_documents([doc['page_content']])
    
    for chunk in chunks:
        metadata = doc['metadata']
        chunk.metadata['url'] = metadata
        chunked_docs.append(chunk)

In [52]:
chunked_docs

Document(page_content='Textbook of\nEngineering Physics\n\n\n\nTextbook of\nEngineering Physics\nDr. P. S. Aithal\nDirector,\nSrinivas Group of Institutions,\nSrinivas Integrated Campus, Mukka\nMangalore, Karnataka\nDr. H. J. Ravindra\nAssistant Professor in Physics,\nSrinivas School of Engineering\nSrinivas Integrated Campus, Mukka\nMangalore, Karnataka', metadata={'url': 'Data/Engineering_Physics.pdf'})

In [53]:
persist_directory_path='./chroma_db_all'

In [54]:
vectordb = Chroma.from_documents(documents=chunked_docs, 
                                 embedding=HuggingFaceEmbeddings(
                                     model_name='BAAI/bge-base-en-v1.5'), 
                                 persist_directory=persist_directory_path)

.gitattributes:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

In [55]:
vectordb.persist()
vectordb=None
os.remove('extracted_data_new.json')
print('done')

done
